In [1]:
import pandas as pd
from pathlib import Path
import pytz
import numpy as np
from scipy.spatial import cKDTree
import datetime


data source: https://mesonet.agron.iastate.edu/request/download.phtml

In [2]:
campd_type_dict = {
    'State': 'string',
    'Facility Name': 'string',
    'Facility ID': 'Int32',
    'Unit ID': 'string',
    'Hour': 'Int8',
    'Operating Time': 'float32',
    'Gross Load (MW)': 'float32',
    'Steam Load (1000 lb/hr)': 'float32',
    'SO2 Mass (lbs)': 'float64',
    'SO2 Mass Measure Indicator': 'string',
    'SO2 Rate (lbs/mmBtu)': 'float32',
    'SO2 Rate Measure Indicator': 'string',
    'CO2 Mass (short tons)': 'float32',
    'CO2 Mass Measure Indicator': 'string',
    'CO2 Rate (short tons/mmBtu)': 'float32',
    'CO2 Rate Measure Indicator': 'string',
    'NOx Mass (lbs)': 'float32',
    'NOx Mass Measure Indicator': 'string',
    'NOx Rate (lbs/mmBtu)': 'float32',
    'NOx Rate Measure Indicator': 'string',
    'Heat Input (mmBtu)': 'float64',
    'Heat Input Measure Indicator': 'string',
    'Primary Fuel Type': 'string',
    'Secondary Fuel Type': 'string',
    'Unit Type': 'string',
    'SO2 Controls': 'string',
    'NOx Controls': 'string',
    'Program Code': 'string',
    'Year': 'int16',
    'Source Category': 'string',
    'Latitude': 'float32',
    'Longitude': 'float32',
    'SO2 Phase': 'string',
    'NOx Phase': 'string',
    'Operating Status': 'string',
    'Max Hourly HI Rate (mmBtu/hr)': 'float32',
    'Associated Generators & Nameplate Capacity (MWe)': 'string',
    'Nameplate Capacity (MWe)': 'float32',
    'Month': 'Int8',
    'hourly_capacity_factor': 'float32',
    'hourly_ramping_factor': 'float32',
    'Nameplate Capacity (MW) - EIA': 'float32',
    'Summer Capacity (MW)': 'float32',
    'Winter Capacity (MW)': 'float32',
    'Minimum Load (MW)': 'float32',
    'Operating Year_min': 'float32',
    'Operating Year_max': 'float32',
    'Operating Year_mean': 'float32',
    'Planned Retirement Year': 'Int16',
    'unit_age_min': 'Int8',
    'unit_age_max': 'Int8',
    'unit_age_mean': 'float32',
    'Day': 'float32'
}

<h2>NY</h2>

In [ ]:
# Concat all temperature csv files into a single df

temp_folder = Path('Temp Data')

df_list = []
for file in temp_folder.iterdir():
    if file.is_file() and file.name.endswith('.csv'):
        df = pd.read_csv(file)
        df_list.append(df)

temp_df = pd.concat(df_list, ignore_index=True)

output_path = Path('Temp Data/temp_2019_2024.csv')
temp_df.to_csv(output_path, index=False)

In [ ]:
temp_df = pd.read_csv("Temp Data/NY/Concatted File/temp_2019_2024.csv")

In [6]:
temp_df = temp_df.rename(columns={"station": "Station", "valid": "datetime", "lon": "Longitude", "lat": "Latitude", "tmpf": "Temperature (F)"})
temp_df

,Station,datetime,Longitude,Latitude,Temperature (F)
0,SWF,2019-01-01 00:45,-74.1048,41.5041,42.8
1,ALB,2019-01-01 00:51,-73.8036,42.7576,41.0
2,SLK,2019-01-01 00:51,-74.2062,44.3853,37.0
3,LGA,2019-01-01 00:51,-73.8803,40.7794,47.0
4,JFK,2019-01-01 00:51,-73.7622,40.6386,49.0
...,...,...,...,...,...
5472405,ITH,2019-12-31 23:56,-76.4584,42.4910,30.90
5472406,HPN,2019-12-31 23:56,-73.7075,41.0669,38.00
5472407,GTB,2019-12-31 23:56,-75.7195,44.0556,31.50
5472408,JRB,2019-12-31 23:56,-74.0090,40.7012,41.00


In [7]:
# replace missing "M" with nan
temp_df.loc[temp_df["Temperature (F)"] == "M", "Temperature (F)"] = np.nan

In [8]:
# Filter out stations that only started reporting after the start date

temp_df["datetime"] = pd.to_datetime(temp_df["datetime"])
first_records = temp_df.groupby("Station")["datetime"].min()
valid_stations = first_records[first_records.dt.date <= datetime.date(2019,1,1)].index
temp_df = temp_df[temp_df["Station"].isin(valid_stations)]


In [ ]:
pd.set_option('display.float_format', lambda x: '%.2f' % x)

temp_df["Temperature (F)"] = pd.to_numeric(temp_df["Temperature (F)"])
temp_df["Temperature (F)"].describe()

In [11]:
# remove unrealistic values for the state
temp_df.loc[temp_df["Temperature (F)"] > 105, "Temperature (F)"] = np.nan

In [ ]:
# round to the lower hour (avoids issues of rounding up from 23 to 00 without changing dates)
temp_df["datetime"] = pd.to_datetime(temp_df["datetime"])
temp_df['datetime'] = temp_df['datetime'].dt.floor('h')
temp_df = temp_df.sort_values(by="datetime")

In [13]:
temp_df.isna().sum()

Station                0
datetime               0
Longitude              0
Latitude               0
Temperature (F)    31322
dtype: int64

In [ ]:
# Interpolate missing temperature values linearly

temp_df = temp_df.sort_values(by=['Station', 'datetime'])

def interpolate_group(df):
    return df.set_index('datetime').interpolate(method='time').reset_index()

temp_df = (
    temp_df.groupby('Station', group_keys=False)
    .apply(interpolate_group)
)

In [15]:
temp_df.isna().sum()

datetime           0
Station            0
Longitude          0
Latitude           0
Temperature (F)    0
dtype: int64

In [16]:
# transform EST/EDT timezone to naive EST (campd is in local standard time without daylight saving)

temp_df['datetime'] = pd.to_datetime(temp_df['datetime'])

# localize to America/New_York 
temp_df['datetime'] = temp_df['datetime'].dt.tz_localize('America/New_York', ambiguous='NaT', nonexistent='shift_forward')

# convert to fixed EST (UTC-5, no daylight saving)
fixed_est = pytz.timezone('Etc/GMT+5') 
temp_df['datetime'] = temp_df['datetime'].dt.tz_convert(fixed_est)

# remove timezone info to make it naive (EST represented as naive time)
temp_df['datetime'] = temp_df['datetime'].dt.tz_localize(None)


In [17]:
facility_df = pd.read_csv("Facility/Concatted File/facility_2019_2024.csv")

In [18]:
facility_df = facility_df[facility_df["State"] == "NY"]
facility_df = facility_df[["State", "Facility Name", "Facility ID", "Longitude", "Latitude"]]
facility_df = facility_df.drop_duplicates(subset=["Facility ID"])
facility_df

,State,Facility Name,Facility ID,Longitude,Latitude
950,NY,Danskammer Generating Station,2480,-73.96,41.57
954,NY,Arthur Kill,2490,-74.20,40.59
957,NY,East River,2493,-73.97,40.73
961,NY,Gowanus Generating Station,2494,-74.01,40.66
993,NY,Hudson Avenue,2496,-73.98,40.71
...,...,...,...,...,...
4110,NY,Valley Energy Center,56940,-74.44,41.41
4319,NY,Momentive Performance Materials,880024,-73.67,42.82
4343,NY,Lehigh Northeast Cement Company,880052,-73.62,43.31
4374,NY,Ravenswood Steam Plant,880100,-73.75,40.76


In [19]:
# Compute closest station mapping for facility data using tree structure

# Unique weather stations with their coords and names
weather_stations = temp_df[['Station', 'Longitude', 'Latitude']].drop_duplicates().reset_index(drop=True)

# Create coords array for k-d tree
weather_coords = weather_stations[['Longitude', 'Latitude']].to_numpy()

# Build k-d tree
weather_tree = cKDTree(weather_coords)

# Facility coords
facility_coords = facility_df[['Longitude', 'Latitude']].drop_duplicates().to_numpy()

# Find nearest station index for each facility location
distances, indices = weather_tree.query(facility_coords)

# Nearest station info
nearest_station_names = weather_stations.loc[indices, 'Station'].values
nearest_station_coords = weather_coords[indices]

# Prepare a DataFrame with mapping info
facility_loc_df = pd.DataFrame(facility_coords, columns=['Longitude', 'Latitude'])
facility_loc_df['Nearest_Station'] = nearest_station_names
facility_loc_df['Station_Longitude'] = nearest_station_coords[:, 0]
facility_loc_df['Station_Latitude'] = nearest_station_coords[:, 1]

# Merge back into full facility_df
facility_df = facility_df.merge(facility_loc_df, on=['Longitude', 'Latitude'], how='left')


In [20]:
campd_ny = pd.read_csv("NY/Hourly/Concatted File/ny_2019_2024_complete_cleaned_Full.csv", dtype=campd_type_dict)
campd_ny["datetime"] = pd.to_datetime(campd_ny["datetime"])

In [21]:
campd_ny.shape

(14028675, 54)

In [26]:
# Merge mapping into campd_ny

campd_ny = campd_ny.merge(facility_df[['Facility ID', 'Nearest_Station']], on='Facility ID', how='left')

In [29]:
campd_ny.shape

(14028675, 55)

In [30]:
duplicates = temp_df[temp_df.duplicated(subset=["Station", "datetime"], keep=False)]
duplicates

,datetime,Station,Longitude,Latitude,Temperature (F)
0,2019-01-01 00:00:00,6B9,-76.44,42.91,46.40
1,2019-01-01 00:00:00,6B9,-76.44,42.91,46.40
2,2019-01-01 00:00:00,6B9,-76.44,42.91,46.40
3,2019-01-01 00:00:00,6B9,-76.44,42.91,46.40
4,2019-01-01 01:00:00,6B9,-76.44,42.91,48.20
...,...,...,...,...,...
122631,2024-12-31 21:00:00,SYR,-76.11,43.11,45.00
122632,2024-12-31 22:00:00,SYR,-76.11,43.11,44.00
122633,2024-12-31 22:00:00,SYR,-76.11,43.11,44.00
122634,2024-12-31 23:00:00,SYR,-76.11,43.11,42.00


In [31]:
temp_df = temp_df.drop_duplicates(subset=["Station", "datetime"])
temp_df

,datetime,Station,Longitude,Latitude,Temperature (F)
0,2019-01-01 00:00:00,6B9,-76.44,42.91,46.40
4,2019-01-01 01:00:00,6B9,-76.44,42.91,48.20
8,2019-01-01 02:00:00,6B9,-76.44,42.91,50.00
12,2019-01-01 03:00:00,6B9,-76.44,42.91,50.00
16,2019-01-01 04:00:00,6B9,-76.44,42.91,51.80
...,...,...,...,...,...
122626,2024-12-31 19:00:00,SYR,-76.11,43.11,47.00
122628,2024-12-31 20:00:00,SYR,-76.11,43.11,47.00
122630,2024-12-31 21:00:00,SYR,-76.11,43.11,45.00
122632,2024-12-31 22:00:00,SYR,-76.11,43.11,44.00


In [32]:
# Merge temp data using station name and timestamp
campd_ny_with_temp = campd_ny.merge(
    temp_df[['Station', 'datetime', 'Temperature (F)']],
    how='left',
    left_on=['Nearest_Station', 'datetime'],
    right_on=['Station', 'datetime']
)

campd_ny_with_temp = campd_ny_with_temp.drop(columns=['Station'])

In [33]:
campd_ny_with_temp["Temperature (F)"].isna().sum()

283072

In [34]:
campd_ny_with_temp.to_csv("NY/Hourly/Concatted File/ny_2019_2024_complete_cleaned_Full.csv", index=False)

In [ ]:
del campd_ny, campd_ny_with_temp
import gc
gc.collect()

<h2>TX</h2>

In [ ]:
# Concat all temperature csv files into a single df

temp_folder = Path('Temp Data/TX')

df_list = []
for file in temp_folder.iterdir():
    if file.is_file() and file.name.endswith('.csv'):
        df = pd.read_csv(file)
        df_list.append(df)

temp_df = pd.concat(df_list, ignore_index=True)

output_path = Path('Temp Data/TX/temp_2019_2024.csv')
temp_df.to_csv(output_path, index=False)

In [5]:
temp_df = pd.read_csv("Temp Data/TX/Concatted File/temp_2019_2024.csv")

In [6]:
temp_df = temp_df.rename(columns={"station": "Station", "valid": "datetime", "lon": "Longitude", "lat": "Latitude", "tmpf": "Temperature (F)"})
temp_df

,Station,datetime,Longitude,Latitude,Temperature (F)
0,GZN,2024-01-01 00:00,-99.0237,32.3658,37.4
1,RYW,2024-01-01 00:15,-97.9659,30.4967,59.0
2,ADS,2024-01-01 00:35,-96.8364,32.9686,38.3
3,FTN,2024-01-01 00:40,-100.0188,28.2086,50.0
4,T35,2024-01-01 00:50,-96.9711,30.8794,50.4
...,...,...,...,...,...
10427431,OCH,2019-12-31 23:56,-94.7094,31.5778,36.0
10427432,BIF,2019-12-31 23:58,-106.3800,31.8495,37.6
10427433,GRK,2019-12-31 23:58,-97.8300,31.0700,46.8
10427434,HLR,2019-12-31 23:58,-97.7145,31.1387,44.6


In [7]:
# Filter out stations that only started reporting after the start date

temp_df["datetime"] = pd.to_datetime(temp_df["datetime"])
first_records = temp_df.groupby("Station")["datetime"].min()
valid_stations = first_records[first_records.dt.date <= datetime.date(2019,1,1)].index
temp_df = temp_df[temp_df["Station"].isin(valid_stations)]


In [8]:
pd.set_option('display.float_format', lambda x: '%.2f' % x)

temp_df["Temperature (F)"] = pd.to_numeric(temp_df["Temperature (F)"])
temp_df["Temperature (F)"].describe()

C:\Users\marce\AppData\Local\Temp\ipykernel_14880\1963221651.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  temp_df["Temperature (F)"] = pd.to_numeric(temp_df["Temperature (F)"])


count   9426270.00
mean         68.29
std          17.16
min         -63.00
25%          57.00
50%          71.00
75%          80.60
max         149.00
Name: Temperature (F), dtype: float64

In [9]:
# lowest temperature ever recorded in Texas is -23 F, fill with NaN if lower than -20
temp_df.loc[temp_df["Temperature (F)"] < -20, "Temperature (F)"] = np.nan

# hottest official temperature recorded in Texas is 120, fill with NaN if higher than 120
temp_df.loc[temp_df["Temperature (F)"] > 120, "Temperature (F)"] = np.nan

In [10]:
pd.set_option('display.float_format', lambda x: '%.2f' % x)

temp_df["Temperature (F)"].describe()

count   9421734.00
mean         68.27
std          17.10
min         -20.00
25%          57.00
50%          71.00
75%          80.60
max         119.00
Name: Temperature (F), dtype: float64

In [ ]:
# round to the lower hour (avoids issues of rounding up from 23 to 00 without changing dates)
temp_df["datetime"] = pd.to_datetime(temp_df["datetime"])
temp_df['datetime'] = temp_df['datetime'].dt.floor('h')
temp_df = temp_df.sort_values(by="datetime")

In [ ]:
# Interpolate missing temperature values linearly

temp_df = temp_df.sort_values(by=['Station', 'datetime'])

def interpolate_group(df):
    return df.set_index('datetime').interpolate(method='time').reset_index()

temp_df = (
    temp_df.groupby('Station', group_keys=False)
    .apply(interpolate_group)
)

In [13]:
# transform CST/CDT timezone to naive CST (campd is in local standard time without daylight saving)

# Ensure datetime column is in datetime format
temp_df['datetime'] = pd.to_datetime(temp_df['datetime'])

# Localize to America/Chicago timezone (handles CST/CDT)
temp_df['datetime'] = temp_df['datetime'].dt.tz_localize('America/Chicago', ambiguous='NaT', nonexistent='shift_forward')

# Convert to fixed CST (UTC-6 without DST)
fixed_cst = pytz.FixedOffset(-6 * 60)  # UTC-6 for CST without DST
temp_df['datetime'] = temp_df['datetime'].dt.tz_convert(fixed_cst)

# Remove timezone information to make it naive (CST represented as naive time)
temp_df['datetime'] = temp_df['datetime'].dt.tz_localize(None)


In [14]:
facility_df = pd.read_csv("Facility/Concatted File/facility_2019_2024.csv")

In [15]:
# Compute closest station mapping for facility data using tree structure

facility_df = facility_df[facility_df["State"] == "TX"]
facility_df = facility_df[["State", "Facility Name", "Facility ID", "Longitude", "Latitude"]]
facility_df = facility_df.drop_duplicates(subset=["Facility ID"])
facility_df

# Unique weather stations with their coords and names
weather_stations = temp_df[['Station', 'Longitude', 'Latitude']].drop_duplicates().reset_index(drop=True)

# Create coords array for k-d tree
weather_coords = weather_stations[['Longitude', 'Latitude']].to_numpy()

# Build k-d tree
weather_tree = cKDTree(weather_coords)

# Facility coords
facility_coords = facility_df[['Longitude', 'Latitude']].drop_duplicates().to_numpy()

# Find nearest station index for each facility location
distances, indices = weather_tree.query(facility_coords)

# Nearest station info
nearest_station_names = weather_stations.loc[indices, 'Station'].values
nearest_station_coords = weather_coords[indices]

# Prepare a DataFrame with mapping info
facility_loc_df = pd.DataFrame(facility_coords, columns=['Longitude', 'Latitude'])
facility_loc_df['Nearest_Station'] = nearest_station_names
facility_loc_df['Station_Longitude'] = nearest_station_coords[:, 0]
facility_loc_df['Station_Latitude'] = nearest_station_coords[:, 1]

# Merge back into full facility_df
facility_df = facility_df.merge(facility_loc_df, on=['Longitude', 'Latitude'], how='left')


In [16]:
facility_df

,State,Facility Name,Facility ID,Longitude,Latitude,Nearest_Station,Station_Longitude,Station_Latitude
0,TX,Copper Station,9,-106.38,31.76,ELP,-106.38,31.81
1,TX,Oklaunion Power Station,127,-99.18,34.08,F05,-99.28,34.23
2,TX,Limestone,298,-96.25,31.42,LXY,-96.51,31.64
3,TX,Laredo,3439,-99.51,27.57,LRD,-99.46,27.54
4,TX,Nueces Bay,3441,-97.42,27.82,CRP,-97.51,27.77
...,...,...,...,...,...,...,...,...
136,TX,SJRR Power Peaking Facility,62548,-95.11,29.88,IAH,-95.36,29.98
137,TX,Braes Bayou Plant,64383,-95.62,29.48,AXH,-95.48,29.51
138,TX,Mark One Power Station,65372,-95.43,29.23,LBX,-95.46,29.11
139,TX,Brotman Power Station,65373,-95.44,29.36,AXH,-95.48,29.51


<h3>NOx</h3>

do it separately due to file size

In [15]:
campd_tx = pd.read_csv("TX/Hourly/Concatted File/Modeling/XGB/NOx/tx_2019_2024_complete_cleaned_Full_manually_encoded_downsampled_only_measured", dtype=campd_type_dict)

In [18]:
# Merge mapping into campd_tx

campd_tx["datetime"] = pd.to_datetime(campd_tx["datetime"])
campd_tx = campd_tx.merge(facility_df[['Facility ID', 'Nearest_Station']], on='Facility ID', how='left')

# Merge temp data using station name and timestamp
campd_tx_with_temp = campd_tx.merge(
    temp_df[['Station', 'datetime', 'Temperature (F)']],
    how='left',
    left_on=['Nearest_Station', 'datetime'],
    right_on=['Station', 'datetime']
)

campd_tx_with_temp = campd_tx_with_temp.drop(columns=['Station'])



In [ ]:
display(campd_tx_with_temp.head(2))
print(campd_tx_with_temp.isna().sum())

In [ ]:
campd_tx_with_temp = campd_tx_with_temp.rename(columns={"Temperature (F)_y": "Temperature (F)"})

In [25]:
campd_tx_with_temp.to_csv("TX/Hourly/Concatted File/Modeling/XGB/NOx/tx_2019_2024_complete_cleaned_Full_manually_encoded_downsampled_only_measured.csv", index=False)

<h3>CO2</h3>

In [3]:
campd_tx = pd.read_csv("TX/Hourly/Concatted File/Modeling/XGB/CO2/tx_2019_2024_complete_cleaned_Full_manually_encoded_downsampled_only_measured", dtype=campd_type_dict)

In [17]:
# Merge mapping into campd_tx

campd_tx["datetime"] = pd.to_datetime(campd_tx["datetime"])
campd_tx = campd_tx.merge(facility_df[['Facility ID', 'Nearest_Station']], on='Facility ID', how='left')

# Merge temp data using station name and timestamp
campd_tx_with_temp = campd_tx.merge(
    temp_df[['Station', 'datetime', 'Temperature (F)']],
    how='left',
    left_on=['Nearest_Station', 'datetime'],
    right_on=['Station', 'datetime']
)

campd_tx_with_temp = campd_tx_with_temp.drop(columns=['Station'])

display(campd_tx_with_temp.head(2))
print(campd_tx_with_temp.isna().sum())


,datetime,State,Facility Name,Facility ID,Unit ID,Date,Hour,Operating Time,Gross Load (MW),Steam Load (1000 lb/hr),...,Missing,TXSO2,Hour_sin,Hour_cos,Day_sin,Day_cos,Month_sin,Month_cos,Nearest_Station,Temperature (F)
0,2019-01-01 04:00:00,TX,Copper Station,9,CTG-1,2019-01-01 00:00:00,4,0.00,0.00,0.00,...,0,0,0.87,0.50,0.78,0.62,0.50,0.87,ELP,32.00
1,2019-01-02 07:00:00,TX,Copper Station,9,CTG-1,2019-01-02 00:00:00,7,0.00,0.00,0.00,...,0,0,0.97,-0.26,0.97,-0.22,0.50,0.87,ELP,25.00


datetime                0
State                   0
Facility Name           0
Facility ID             0
Unit ID                 0
                    ...  
Day_cos                 0
Month_sin               0
Month_cos               0
Nearest_Station         0
Temperature (F)    451479
Length: 75, dtype: int64


In [18]:
campd_tx_with_temp.to_csv("TX/Hourly/Concatted File/Modeling/XGB/CO2/tx_2019_2024_complete_cleaned_Full_manually_encoded_downsampled_only_measured.csv", index=False)


<h3>SO2</h3>

In [19]:
campd_tx = pd.read_csv("TX/Hourly/Concatted File/Modeling/XGB/SO2/tx_2019_2024_complete_cleaned_Full_manually_encoded_downsampled_only_measured", dtype=campd_type_dict)

FileNotFoundError: [Errno 2] No such file or directory: 'TX/Hourly/Concatted File/Modeling/XGB/SO2/tx_2019_2024_complete_cleaned_Full_manually_encoded_downsampled_only_measured'

In [ ]:
# Merge mapping into campd_tx

campd_tx["datetime"] = pd.to_datetime(campd_tx["datetime"])
campd_tx = campd_tx.merge(facility_df[['Facility ID', 'Nearest_Station']], on='Facility ID', how='left')

# Merge temp data using station name and timestamp
campd_tx_with_temp = campd_tx.merge(
    temp_df[['Station', 'datetime', 'Temperature (F)']],
    how='left',
    left_on=['Nearest_Station', 'datetime'],
    right_on=['Station', 'datetime']
)

campd_tx_with_temp = campd_tx_with_temp.drop(columns=['Station'])

display(campd_tx_with_temp.head(2))
print(campd_tx_with_temp.isna().sum())


In [ ]:
campd_tx_with_temp.to_csv("TX/Hourly/Concatted File/Modeling/XGB/SO2/tx_2019_2024_complete_cleaned_Full_manually_encoded_downsampled_only_measured.csv", index=False)


<h2>WY</h2>

In [24]:
# Concat all temperature csv files into a single df

temp_folder = Path('Temp Data/WY')

df_list = []
for file in temp_folder.iterdir():
    if file.is_file() and file.name.endswith('.csv'):
        df = pd.read_csv(file)
        df_list.append(df)

temp_df = pd.concat(df_list, ignore_index=True)

output_path = Path('Temp Data/WY/temp_2019_2024.csv')
temp_df.to_csv(output_path, index=False)

In [25]:
temp_df = pd.read_csv("Temp Data/WY/temp_2019_2024.csv")

In [26]:
temp_df = temp_df.rename(columns={"station": "Station", "valid": "datetime", "lon": "Longitude", "lat": "Latitude", "tmpf": "Temperature (F)"})
temp_df

,Station,datetime,Longitude,Latitude,Temperature (F)
0,TBX,2024-01-01 00:50,-108.24,43.46,35.60
1,SHR,2024-01-01 00:53,-106.97,44.77,22.00
2,LAR,2024-01-01 00:53,-105.67,41.31,18.00
3,CPR,2024-01-01 00:53,-106.46,42.91,32.00
4,CYS,2024-01-01 00:53,-104.81,41.16,26.00
...,...,...,...,...,...
3854849,GUR,2023-12-31 23:55,-104.73,42.26,20.00
3854850,HSG,2023-12-31 23:55,-108.39,43.71,24.00
3854851,COD,2023-12-31 23:56,-109.02,44.52,32.00
3854852,JAC,2023-12-31 23:56,-110.74,43.61,8.00


In [27]:
# Filter out stations that only started reporting after the start date

temp_df["datetime"] = pd.to_datetime(temp_df["datetime"])
first_records = temp_df.groupby("Station")["datetime"].min()
valid_stations = first_records[first_records.dt.date <= datetime.date(2019,1,1)].index
temp_df = temp_df[temp_df["Station"].isin(valid_stations)]


In [28]:
pd.set_option('display.float_format', lambda x: '%.2f' % x)

temp_df["Temperature (F)"] = pd.to_numeric(temp_df["Temperature (F)"])
temp_df["Temperature (F)"].describe()

C:\Users\marce\AppData\Local\Temp\ipykernel_17856\1963221651.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  temp_df["Temperature (F)"] = pd.to_numeric(temp_df["Temperature (F)"])


count   3537816.00
mean         44.10
std          21.55
min         -42.00
25%          28.40
50%          42.80
75%          60.00
max         107.00
Name: Temperature (F), dtype: float64

In [ ]:
# values appear accurate, no unrealistic outliers

In [29]:
# round to the lower hour (avoids issues of rounding up from 23 to 00 without changing dates)
temp_df["datetime"] = pd.to_datetime(temp_df["datetime"])
temp_df['datetime'] = temp_df['datetime'].dt.floor('h')
temp_df = temp_df.sort_values(by="datetime")
temp_df

C:\Users\marce\AppData\Local\Temp\ipykernel_17856\1182388849.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  temp_df["datetime"] = pd.to_datetime(temp_df["datetime"])
C:\Users\marce\AppData\Local\Temp\ipykernel_17856\1182388849.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  temp_df['datetime'] = temp_df['datetime'].dt.floor('h')


,Station,datetime,Longitude,Latitude,Temperature (F)
2252348,BPI,2019-01-01 00:00:00,-110.11,42.59,-18.90
2252351,BYG,2019-01-01 00:00:00,-106.72,44.38,-4.00
2252352,GCC,2019-01-01 00:00:00,-105.54,44.35,-11.90
2252353,LND,2019-01-01 00:00:00,-108.73,42.82,-0.90
2252354,CPR,2019-01-01 00:00:00,-106.46,42.91,-4.00
...,...,...,...,...,...
324902,TBX,2024-12-31 23:00:00,-108.24,43.46,NaN
324901,ECS,2024-12-31 23:00:00,-104.32,43.89,19.40
324900,POY,2024-12-31 23:00:00,-108.79,44.87,17.60
324907,U68,2024-12-31 23:00:00,-108.45,44.91,21.20


In [ ]:
# Interpolate missing temperature values linearly

temp_df = temp_df.sort_values(by=['Station', 'datetime'])

def interpolate_group(df):
    return df.set_index('datetime').interpolate(method='time').reset_index()

temp_df = (
    temp_df.groupby('Station', group_keys=False)
    .apply(interpolate_group)
)

In [31]:
# transform MST/MDT timezone to naive MST (campd is in local standard time without daylight saving)

# Ensure datetime column is in datetime format
temp_df['datetime'] = pd.to_datetime(temp_df['datetime'])

# Localize to America/Denver timezone (handles MST/MDT with DST)
temp_df['datetime'] = temp_df['datetime'].dt.tz_localize('America/Denver', ambiguous='NaT', nonexistent='shift_forward')

# Convert to fixed MST (UTC-7 without DST)
fixed_mst = pytz.FixedOffset(-7 * 60)  # UTC-7 for MST without DST
temp_df['datetime'] = temp_df['datetime'].dt.tz_convert(fixed_mst)

# Remove timezone information to make it naive (MST represented as naive time)
temp_df['datetime'] = temp_df['datetime'].dt.tz_localize(None)


In [32]:
facility_df = pd.read_csv("Facility/Concatted File/facility_2019_2024.csv")

In [33]:
# Compute closest station mapping for facility data using tree structure

facility_df = facility_df[facility_df["State"] == "WY"]
facility_df = facility_df[["State", "Facility Name", "Facility ID", "Longitude", "Latitude"]]
facility_df = facility_df.drop_duplicates(subset=["Facility ID"])
facility_df

# Unique weather stations with their coords and names
weather_stations = temp_df[['Station', 'Longitude', 'Latitude']].drop_duplicates().reset_index(drop=True)

# Create coords array for k-d tree
weather_coords = weather_stations[['Longitude', 'Latitude']].to_numpy()

# Build k-d tree
weather_tree = cKDTree(weather_coords)

# Facility coords
facility_coords = facility_df[['Longitude', 'Latitude']].drop_duplicates().to_numpy()

# Find nearest station index for each facility location
distances, indices = weather_tree.query(facility_coords)

# Nearest station info
nearest_station_names = weather_stations.loc[indices, 'Station'].values
nearest_station_coords = weather_coords[indices]

# Prepare a DataFrame with mapping info
facility_loc_df = pd.DataFrame(facility_coords, columns=['Longitude', 'Latitude'])
facility_loc_df['Nearest_Station'] = nearest_station_names
facility_loc_df['Station_Longitude'] = nearest_station_coords[:, 0]
facility_loc_df['Station_Latitude'] = nearest_station_coords[:, 1]

# Merge back into full facility_df
facility_df = facility_df.merge(facility_loc_df, on=['Longitude', 'Latitude'], how='left')


In [34]:
campd_wy = pd.read_csv("WY/Hourly/Concatted File/Modeling/XGB/NOx/wy_2019_2024_complete_cleaned_Full_manually_encoded", dtype=campd_type_dict)

In [35]:
# Merge mapping into campd_wy

campd_wy["datetime"] = pd.to_datetime(campd_wy["datetime"])
campd_wy = campd_wy.merge(facility_df[['Facility ID', 'Nearest_Station']], on='Facility ID', how='left')

# Merge temp data using station name and timestamp
campd_wy_with_temp = campd_wy.merge(
    temp_df[['Station', 'datetime', 'Temperature (F)']],
    how='left',
    left_on=['Nearest_Station', 'datetime'],
    right_on=['Station', 'datetime']
)

campd_wy_with_temp = campd_wy_with_temp.drop(columns=['Station'])

display(campd_wy_with_temp.head(2))
print(campd_wy_with_temp.isna().sum())


,datetime,State,Facility Name,Facility ID,Unit ID,Date,Hour,Operating Time,Gross Load (MW),SO2 Mass (lbs),...,ARP,MATS,Hour_sin,Hour_cos,Day_sin,Day_cos,Month_sin,Month_cos,Nearest_Station,Temperature (F)
0,2019-01-01,WY,Dave Johnston,4158,BW41,2019-01-01 00:00:00,0,1.00,103.00,728.40,...,1,1,0.00,1.00,0.78,0.62,0.50,0.87,DGW,-0.90
1,2019-01-01,WY,Dave Johnston,4158,BW41,2019-01-01 00:00:00,0,1.00,103.00,728.40,...,1,1,0.00,1.00,0.78,0.62,0.50,0.87,DGW,-0.90


datetime               0
State                  0
Facility Name          0
Facility ID            0
Unit ID                0
                   ...  
Day_cos                0
Month_sin              0
Month_cos              0
Nearest_Station        0
Temperature (F)    91581
Length: 79, dtype: int64


In [36]:
campd_wy_with_temp.to_csv("WY/Hourly/Concatted File/Modeling/XGB/NOx/wy_2019_2024_complete_cleaned_Full_manually_encoded", index=False)


<h2>ND</h2>

In [38]:
# Concat all temperature csv files into a single df

temp_folder = Path('Temp Data/ND')

df_list = []
for file in temp_folder.iterdir():
    if file.is_file() and file.name.endswith('.csv'):
        df = pd.read_csv(file)
        df_list.append(df)

temp_df = pd.concat(df_list, ignore_index=True)

output_path = Path('Temp Data/ND/temp_2019_2024.csv')
temp_df.to_csv(output_path, index=False)

In [39]:
temp_df = temp_df.rename(columns={"station": "Station", "valid": "datetime", "lon": "Longitude", "lat": "Latitude", "tmpf": "Temperature (F)"})
temp_df

,Station,datetime,Longitude,Latitude,Temperature (F)
0,BIS,2019-01-01 00:52,-100.75,46.77,-17.00
1,ISN,2019-01-01 00:52,-103.64,48.17,-9.00
2,N60,2019-01-01 00:52,-101.44,47.65,-6.00
3,GFK,2019-01-01 00:53,-97.18,47.95,-18.90
4,HEI,2019-01-01 00:53,-102.66,46.01,-14.10
...,...,...,...,...,...
5696707,7L2,2024-12-31 23:55,-100.25,46.22,19.40
5696708,46D,2024-12-31 23:55,-99.15,47.45,14.00
5696709,DVL,2024-12-31 23:56,-98.91,48.11,10.00
5696710,DIK,2024-12-31 23:56,-102.80,46.80,17.00


In [40]:
# Filter out stations that only started reporting after the start date

temp_df["datetime"] = pd.to_datetime(temp_df["datetime"])
first_records = temp_df.groupby("Station")["datetime"].min()
valid_stations = first_records[first_records.dt.date <= datetime.date(2019,1,1)].index
temp_df = temp_df[temp_df["Station"].isin(valid_stations)]


In [41]:
pd.set_option('display.float_format', lambda x: '%.2f' % x)

temp_df["Temperature (F)"] = pd.to_numeric(temp_df["Temperature (F)"])
temp_df["Temperature (F)"].describe()

C:\Users\marce\AppData\Local\Temp\ipykernel_17856\1963221651.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  temp_df["Temperature (F)"] = pd.to_numeric(temp_df["Temperature (F)"])


count   5149557.00
mean         41.59
std          24.93
min         -41.00
25%          24.10
50%          42.00
75%          62.00
max         120.20
Name: Temperature (F), dtype: float64

In [42]:
# hottest official temperature recorded in ND is 121, fill with NaN if higher than 120
temp_df.loc[temp_df["Temperature (F)"] > 120, "Temperature (F)"] = np.nan

In [43]:
# round to the lower hour (avoids issues of rounding up from 23 to 00 without changing dates)
temp_df["datetime"] = pd.to_datetime(temp_df["datetime"])
temp_df['datetime'] = temp_df['datetime'].dt.floor('h')
temp_df = temp_df.sort_values(by="datetime")
temp_df

C:\Users\marce\AppData\Local\Temp\ipykernel_17856\1182388849.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  temp_df["datetime"] = pd.to_datetime(temp_df["datetime"])
C:\Users\marce\AppData\Local\Temp\ipykernel_17856\1182388849.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  temp_df['datetime'] = temp_df['datetime'].dt.floor('h')


,Station,datetime,Longitude,Latitude,Temperature (F)
0,BIS,2019-01-01 00:00:00,-100.75,46.77,-17.00
3797808,BIS,2019-01-01 00:00:00,-100.75,46.77,-17.00
3797812,HEI,2019-01-01 00:00:00,-102.66,46.01,-14.10
3797813,FAR,2019-01-01 00:00:00,-96.81,46.93,-16.10
3797814,MOT,2019-01-01 00:00:00,-101.28,48.26,-5.00
...,...,...,...,...,...
1898900,46D,2024-12-31 23:00:00,-99.15,47.45,14.00
1898901,DVL,2024-12-31 23:00:00,-98.91,48.11,10.00
1898902,DIK,2024-12-31 23:00:00,-102.80,46.80,17.00
3797807,JMS,2024-12-31 23:00:00,-98.68,46.93,15.00


In [ ]:
# Interpolate missing temperature values linearly

temp_df = temp_df.sort_values(by=['Station', 'datetime'])

def interpolate_group(df):
    return df.set_index('datetime').interpolate(method='time').reset_index()

temp_df = (
    temp_df.groupby('Station', group_keys=False)
    .apply(interpolate_group)
)

In [45]:
# transform MST/MDT timezone to naive MST (campd is in local standard time without daylight saving)

# Ensure datetime column is in datetime format
temp_df['datetime'] = pd.to_datetime(temp_df['datetime'])

# Localize to America/Denver timezone (handles MST/MDT with DST)
temp_df['datetime'] = temp_df['datetime'].dt.tz_localize('America/Denver', ambiguous='NaT', nonexistent='shift_forward')

# Convert to fixed MST (UTC-7 without DST)
fixed_mst = pytz.FixedOffset(-7 * 60)  # UTC-7 for MST without DST
temp_df['datetime'] = temp_df['datetime'].dt.tz_convert(fixed_mst)

# Remove timezone information to make it naive (MST represented as naive time)
temp_df['datetime'] = temp_df['datetime'].dt.tz_localize(None)


In [46]:
facility_df = pd.read_csv("Facility/Concatted File/facility_2019_2024_ND.csv")

In [47]:
# Compute closest station mapping for facility data using tree structure

facility_df = facility_df[facility_df["State"] == "ND"]
facility_df = facility_df[["State", "Facility ID", "Longitude", "Latitude"]]
facility_df = facility_df.drop_duplicates(subset=["Facility ID"])
facility_df

# Unique weather stations with their coords and names
weather_stations = temp_df[['Station', 'Longitude', 'Latitude']].drop_duplicates().reset_index(drop=True)

# Create coords array for k-d tree
weather_coords = weather_stations[['Longitude', 'Latitude']].to_numpy()

# Build k-d tree
weather_tree = cKDTree(weather_coords)

# Facility coords
facility_coords = facility_df[['Longitude', 'Latitude']].drop_duplicates().to_numpy()

# Find nearest station index for each facility location
distances, indices = weather_tree.query(facility_coords)

# Nearest station info
nearest_station_names = weather_stations.loc[indices, 'Station'].values
nearest_station_coords = weather_coords[indices]

# Prepare a DataFrame with mapping info
facility_loc_df = pd.DataFrame(facility_coords, columns=['Longitude', 'Latitude'])
facility_loc_df['Nearest_Station'] = nearest_station_names
facility_loc_df['Station_Longitude'] = nearest_station_coords[:, 0]
facility_loc_df['Station_Latitude'] = nearest_station_coords[:, 1]

# Merge back into full facility_df
facility_df = facility_df.merge(facility_loc_df, on=['Longitude', 'Latitude'], how='left')


In [59]:
campd_nd = pd.read_csv("ND/Hourly/Concatted File/nd_2019_2024_complete_cleaned.csv", dtype=campd_type_dict)


In [ ]:
campd_nd["datetime"] = pd.to_datetime(campd_nd["Date"].astype(str) + ' ' + campd_nd["Hour"].astype(str).str.zfill(2) + ":00:00")

In [63]:
# Merge mapping into campd_nd

campd_nd["datetime"] = pd.to_datetime(campd_nd["datetime"])
campd_nd = campd_nd.merge(facility_df[['Facility ID', 'Nearest_Station']], on='Facility ID', how='left')

# Merge temp data using station name and timestamp
campd_nd_with_temp = campd_nd.merge(
    temp_df[['Station', 'datetime', 'Temperature (F)']],
    how='left',
    left_on=['Nearest_Station', 'datetime'],
    right_on=['Station', 'datetime']
)

campd_nd_with_temp = campd_nd_with_temp.drop(columns=['Station'])

display(campd_nd_with_temp.head(2))
print(campd_nd_with_temp.isna().sum())


,State,Facility Name,Facility ID,Unit ID,Date,Hour,Operating Time,Gross Load (MW),Steam Load (1000 lb/hr),SO2 Mass (lbs),...,Operating Year_min,Operating Year_max,Operating Year_mean,Planned Retirement Year,unit_age_min,unit_age_max,unit_age_mean,datetime,Nearest_Station,Temperature (F)
0,ND,R M Heskett,2790,B2,2019-01-01,0,1.00,59.00,NaN,251.70,...,NaN,NaN,NaN,<NA>,<NA>,<NA>,NaN,2019-01-01 00:00:00,Y19,-13.00
1,ND,R M Heskett,2790,B2,2019-01-01,1,1.00,59.00,NaN,264.70,...,NaN,NaN,NaN,<NA>,<NA>,<NA>,NaN,2019-01-01 01:00:00,Y19,-11.20


State                                                     0
Facility Name                                             0
Facility ID                                               0
Unit ID                                                   0
Date                                                      0
Hour                                                      0
Operating Time                                         1021
Gross Load (MW)                                           0
Steam Load (1000 lb/hr)                             1013673
SO2 Mass (lbs)                                       352829
SO2 Mass Measure Indicator                           352829
SO2 Rate (lbs/mmBtu)                                 352829
SO2 Rate Measure Indicator                                0
CO2 Mass (short tons)                                352829
CO2 Mass Measure Indicator                           352829
CO2 Rate (short tons/mmBtu)                          352829
CO2 Rate Measure Indicator              

In [64]:
campd_nd_with_temp.to_csv("ND/Hourly/Concatted File/nd_2019_2024_complete_cleaned.csv", index=False)


<h2>MT</h2>

In [3]:
# Concat all temperature csv files into a single df

temp_folder = Path('Temp Data/MT')

df_list = []
for file in temp_folder.iterdir():
    if file.is_file() and file.name.endswith('.csv'):
        df = pd.read_csv(file)
        df_list.append(df)

temp_df = pd.concat(df_list, ignore_index=True)

output_path = Path('Temp Data/MT/temp_2019_2024.csv')
temp_df.to_csv(output_path, index=False)

In [4]:
temp_df = temp_df.rename(columns={"station": "Station", "valid": "datetime", "lon": "Longitude", "lat": "Latitude", "tmpf": "Temperature (F)"})
temp_df

,Station,datetime,Longitude,Latitude,Temperature (F)
0,JDN,2019-01-01 00:48,-106.9475,47.3258,-5.0
1,BHK,2019-01-01 00:51,-104.2595,46.3476,-13.0
2,DLN,2019-01-01 00:53,-112.5525,45.2554,-9.0
3,LVM,2019-01-01 00:53,-110.4500,45.7000,-8.0
4,MLS,2019-01-01 00:53,-105.8863,46.4280,0.0
...,...,...,...,...,...
5820337,1GM,2024-12-31 23:55,-112.1000,47.3300,20.7
5820338,GDV,2024-12-31 23:56,-104.8072,47.1387,18.0
5820339,CTB,2024-12-31 23:56,-112.3761,48.6084,18.0
5820340,BZN,2024-12-31 23:56,-111.1608,45.7881,14.0


In [5]:
pd.set_option('display.float_format', lambda x: '%.2f' % x)

temp_df["Temperature (F)"] = pd.to_numeric(temp_df["Temperature (F)"])
temp_df["Temperature (F)"].describe()

count   5780202.00
mean         44.52
std          22.00
min         -45.00
25%          30.00
50%          44.60
75%          60.00
max         109.00
Name: Temperature (F), dtype: float64

In [6]:
# round to the lower hour (avoids issues of rounding up from 23 to 00 without changing dates)
temp_df["datetime"] = pd.to_datetime(temp_df["datetime"])
temp_df['datetime'] = temp_df['datetime'].dt.floor('h')
temp_df = temp_df.sort_values(by="datetime")
temp_df

,Station,datetime,Longitude,Latitude,Temperature (F)
0,JDN,2019-01-01 00:00:00,-106.95,47.33,-5.00
1940115,BHK,2019-01-01 00:00:00,-104.26,46.35,-13.00
1940116,DLN,2019-01-01 00:00:00,-112.55,45.26,-9.00
1940117,LVM,2019-01-01 00:00:00,-110.45,45.70,-8.00
1940118,MLS,2019-01-01 00:00:00,-105.89,46.43,0.00
...,...,...,...,...,...
1940088,MSO,2024-12-31 23:00:00,-114.09,46.92,26.00
3880227,SDY,2024-12-31 23:00:00,-104.19,47.71,16.00
3880226,BZN,2024-12-31 23:00:00,-111.16,45.79,14.00
1940099,GPI,2024-12-31 23:00:00,-114.26,48.30,27.00


In [7]:
# Filter out stations that only started reporting after the start date

temp_df["datetime"] = pd.to_datetime(temp_df["datetime"])
first_records = temp_df.groupby("Station")["datetime"].min()
valid_stations = first_records[first_records.dt.date <= datetime.date(2019,1,1)].index
temp_df = temp_df[temp_df["Station"].isin(valid_stations)]


In [ ]:
# Interpolate missing temperature values linearly

temp_df = temp_df.sort_values(by=['Station', 'datetime'])

def interpolate_group(df):
    return df.set_index('datetime').interpolate(method='time').reset_index()

temp_df = (
    temp_df.groupby('Station', group_keys=False)
    .apply(interpolate_group)
)

In [9]:
# transform MST/MDT timezone to naive MST (campd is in local standard time without daylight saving)

# Ensure datetime column is in datetime format
temp_df['datetime'] = pd.to_datetime(temp_df['datetime'])

# Localize to America/Denver timezone (handles MST/MDT with DST)
temp_df['datetime'] = temp_df['datetime'].dt.tz_localize('America/Denver', ambiguous='NaT', nonexistent='shift_forward')

# Convert to fixed MST (UTC-7 without DST)
fixed_mst = pytz.FixedOffset(-7 * 60)  # UTC-7 for MST without DST
temp_df['datetime'] = temp_df['datetime'].dt.tz_convert(fixed_mst)

# Remove timezone information to make it naive (MST represented as naive time)
temp_df['datetime'] = temp_df['datetime'].dt.tz_localize(None)


In [10]:
facility_df = pd.read_csv("Facility/Concatted File/facility_2019_2024_MT.csv")

In [11]:
# Compute closest station mapping for facility data using tree structure

facility_df = facility_df[facility_df["State"] == "MT"]
facility_df = facility_df[["State", "Facility ID", "Longitude", "Latitude"]]
facility_df = facility_df.drop_duplicates(subset=["Facility ID"])
facility_df

# Unique weather stations with their coords and names
weather_stations = temp_df[['Station', 'Longitude', 'Latitude']].drop_duplicates().reset_index(drop=True)

# Create coords array for k-d tree
weather_coords = weather_stations[['Longitude', 'Latitude']].to_numpy()

# Build k-d tree
weather_tree = cKDTree(weather_coords)

# Facility coords
facility_coords = facility_df[['Longitude', 'Latitude']].drop_duplicates().to_numpy()

# Find nearest station index for each facility location
distances, indices = weather_tree.query(facility_coords)

# Nearest station info
nearest_station_names = weather_stations.loc[indices, 'Station'].values
nearest_station_coords = weather_coords[indices]

# Prepare a DataFrame with mapping info
facility_loc_df = pd.DataFrame(facility_coords, columns=['Longitude', 'Latitude'])
facility_loc_df['Nearest_Station'] = nearest_station_names
facility_loc_df['Station_Longitude'] = nearest_station_coords[:, 0]
facility_loc_df['Station_Latitude'] = nearest_station_coords[:, 1]

# Merge back into full facility_df
facility_df = facility_df.merge(facility_loc_df, on=['Longitude', 'Latitude'], how='left')


In [12]:
campd_mt = pd.read_csv("MT/Hourly/Concatted File/mt_2019_2024_complete_cleaned.csv", dtype=campd_type_dict)


In [13]:
campd_mt["datetime"] = pd.to_datetime(campd_mt["Date"].astype(str) + ' ' + campd_mt["Hour"].astype(str).str.zfill(2) + ":00:00")


In [17]:
# Merge mapping into campd_mt

campd_mt["datetime"] = pd.to_datetime(campd_mt["datetime"])
campd_mt = campd_mt.merge(facility_df[['Facility ID', 'Nearest_Station']], on='Facility ID', how='left')

# Merge temp data using station name and timestamp
campd_mt_with_temp = campd_mt.merge(
    temp_df[['Station', 'datetime', 'Temperature (F)']],
    how='left',
    left_on=['Nearest_Station', 'datetime'],
    right_on=['Station', 'datetime']
)

campd_mt_with_temp = campd_mt_with_temp.drop(columns=['Station'])

display(campd_mt_with_temp.head(2))
print(campd_mt_with_temp.isna().sum())


,State,Facility Name,Facility ID,Unit ID,Date,Hour,Operating Time,Gross Load (MW),Steam Load (1000 lb/hr),SO2 Mass (lbs),...,Operating Year_mean,Planned Retirement Year,unit_age_min,unit_age_max,unit_age_mean,datetime,Nearest_Station_x,Nearest_Station_y,Nearest_Station,Temperature (F)
0,MT,Glendive Generating Station,2176,GT-2,2019-01-01,0,0.00,0.00,NaN,NaN,...,2003.00,<NA>,16,16,16.00,2019-01-01,GDV,GDV,GDV,-9.00
1,MT,Glendive Generating Station,2176,GT-2,2019-01-01,0,0.00,0.00,NaN,NaN,...,2003.00,<NA>,16,16,16.00,2019-01-01,GDV,GDV,GDV,-9.00


State                                                     0
Facility Name                                             0
Facility ID                                               0
Unit ID                                                   0
Date                                                      0
Hour                                                      0
Operating Time                                            0
Gross Load (MW)                                           0
Steam Load (1000 lb/hr)                             1854305
SO2 Mass (lbs)                                       916969
SO2 Mass Measure Indicator                           916969
SO2 Rate (lbs/mmBtu)                                 916969
SO2 Rate Measure Indicator                                0
CO2 Mass (short tons)                                916969
CO2 Mass Measure Indicator                           916969
CO2 Rate (short tons/mmBtu)                          916969
CO2 Rate Measure Indicator              

In [23]:
campd_mt_with_temp.to_csv("MT/Hourly/Concatted File/mt_2019_2024_complete_cleaned.csv", index=False)
